# The Slope Chart Principle: 25 Years of World Urbanization

**Companion notebook for the Medium article:**  
[The Slope Chart Principle: How To Reveal The Surprising Truth Behind 25 Years Of World Urbanization](https://medium.com/@loewenj700)

Two charts built with Python + Plotly from the World Bank urban population dataset (1960–2025).

**Dataset:** `urban_pct_all.csv` — urban population as a share of total population, all World Bank countries and regions, 1960–2025  
**Source:** [World Bank Indicators API](https://api.worldbank.org/v2/country/all/indicator/SP.URB.TOTL.IN.ZS) — indicator SP.URB.TOTL.IN.ZS  
**Author:** John Loewen | [Medium](https://medium.com/@loewenj700) | [GitHub](https://github.com/loewenj700)

---

## Setup

Install dependencies if needed:

```
pip install pandas plotly
```

In [ ]:
import pandas as pd
import plotly.graph_objects as go

## The Dataset

Load `urban_pct_all.csv` — 262 unique country and aggregate codes from 1960 to 2025.

Columns we use:

- `iso3` — the World Bank three-letter country or region code
- `country` — the entity name
- `year` — the reference year (1960–2025)
- `value` — urban population as a percentage of total population

In [ ]:
df = pd.read_csv('urban_pct_all.csv')
print(df.shape)
print(df.head())

---

## Chart 1: The Slope Chart In Action

Compare the six standard World Bank regional aggregates at 2000 and 2025 — the before-and-after structure the slope chart was built for.

**Key findings:**
- East Asia & Pacific climbed from 41.6% to 64.8%, a 23-point rise that dwarfs every other region.
- Sub-Saharan Africa (32.7% → 44.7%) was already ahead of South Asia (27.0% → 35.8%) in 2000 and widened that lead to 8.9 points by 2025 — the opposite of what most readers expect given the density of South Asian cities.
- North America, Latin America, and Europe barely moved; those regions were already highly urbanized.

In [ ]:
region_iso = ['SAS', 'SSF', 'EAS', 'LCN', 'ECS', 'NAC']
region_names = {
    'SAS': 'South Asia',
    'SSF': 'Sub-Saharan Africa',
    'EAS': 'East Asia & Pacific',
    'LCN': 'Latin America & Caribbean',
    'ECS': 'Europe & Central Asia',
    'NAC': 'North America',
}
highlight_colors = {
    'East Asia & Pacific': '#D62728',
    'Sub-Saharan Africa': '#E69F00',
    'South Asia': '#0072B2',
}

regions = df[df['iso3'].isin(region_iso)].copy()
regions['label'] = regions['iso3'].map(region_names)

slope_df = regions[regions['year'].isin([2000, 2025])].pivot(
    index='label', columns='year', values='value'
).reset_index()
slope_df.columns = ['label', 'v2000', 'v2025']

fig = go.Figure()
for _, row in slope_df.iterrows():
    color = highlight_colors.get(row['label'], '#B8B8B8')
    width = 3 if row['label'] in highlight_colors else 1.5
    fig.add_trace(go.Scatter(
        x=[2000, 2025],
        y=[row['v2000'], row['v2025']],
        mode='lines+markers',
        line=dict(color=color, width=width),
        marker=dict(size=8, color=color),
        showlegend=False,
    ))
    if row['label'] in highlight_colors:
        fig.add_annotation(
            x=2025, y=row['v2025'],
            text=f"{row['label']} {row['v2025']:.1f}%",
            xanchor='left', xshift=12,
            showarrow=False,
            font=dict(color=color, size=13),
        )

fig.update_layout(
    title='East Asia urbanized fastest \u2014 Sub-Saharan Africa widened its lead over South Asia',
    xaxis=dict(tickvals=[2000, 2025], ticktext=['2000', '2025']),
    yaxis=dict(title='Urban share (%)', ticksuffix='%'),
    margin=dict(r=280),
)
fig.show()

---

## Chart 2: The Long Arc

The slope chart answers "who changed and by how much?" — but cannot show the full history. For that, a line chart covering the whole 1960–2025 arc.

**Key finding:** Global urban share crossed 50% around 2007 — the year we became, for the first time in history, a majority-urban species. By 2025 it stands at 57.8%.

In [ ]:
world = df[df['iso3'] == 'WLD'].sort_values('year')
world_line = world[world['year'].between(1960, 2025)]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=world_line['year'],
    y=world_line['value'],
    mode='lines',
    line=dict(color='#D62728', width=3),
))
fig.add_hline(y=50, line_dash='dash', line_color='gray', line_width=1.5)
fig.add_annotation(
    x=1990, y=50, text='50% threshold',
    xanchor='center', yshift=14,
    showarrow=False,
    font=dict(size=13, color='gray'),
)
fig.update_layout(
    title='More than half the world now lives in cities',
    xaxis_title='Year',
    yaxis=dict(title='Urban share (%)', ticksuffix='%'),
)
fig.show()

---

**Source:** World Bank Indicators API, SP.URB.TOTL.IN.ZS — urban population as a share of total population.  
**Article:** [The Slope Chart Principle: How To Reveal The Surprising Truth Behind 25 Years Of World Urbanization](https://medium.com/@loewenj700)  
**Author:** John Loewen — [Medium](https://medium.com/@loewenj700) | [Substack](https://johnloewen.substack.com/)